# Аналитик отзывов для бизнеса

Гибридный ML + AI проект: классификация тональности отзывов клиентов и автоматическая генерация бизнес-рекомендаций на основе жалоб.

Датасет: [Amazon Fine Food Reviews (Kaggle)](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews)

In [29]:
!pip install -q fastapi "uvicorn[standard]" scikit-learn joblib pandas requests openai


In [30]:
import os
import time
import subprocess

os.makedirs("review-api", exist_ok=True)
print("Готово. Папка проекта: review-api")


Готово. Папка проекта: review-api


## Блок 1 — Данные: Amazon Fine Food Reviews

**Как получить `kaggle.json`:**
1. Зайдите на [kaggle.com](https://www.kaggle.com) → аватар → **Settings**
2. Раздел **API** → кнопка **Create New Token** — скачается файл `kaggle.json`
3. Запустите следующую ячейку и загрузите этот файл в открывшемся диалоге


In [31]:
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle-креды подхвачены из Colab Secrets.")


Kaggle-креды подхвачены из Colab Secrets.


In [32]:
!kaggle datasets download -d snap/amazon-fine-food-reviews -p data --unzip
!ls -la data


Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:12<00:00, 20.8MB/s]

total 657940
drwxr-xr-x 2 root root      4096 Sep 17 12:01 .
drwxr-xr-x 1 root root      4096 Sep 17 10:28 ..
-rw-r--r-- 1 root root 372798464 Sep 17 12:01 database.sqlite
-rw-r--r-- 1 root root       277 Sep 17 12:01 hashes.txt
-rw-r--r-- 1 root root 300904694 Sep 17 12:01 Reviews.csv


In [33]:
import pandas as pd

df = pd.read_csv("data/Reviews.csv")
print(df.shape)
df.head()


(568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## Блок 2 — Модель как артефакт

Готовим данные, обучаем, сохраняем как файл.


In [34]:
df = df.dropna(subset=["Text", "Score"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["UserId", "Time", "Text"]).reset_index(drop=True)


def score_to_sentiment(score):
    if score <= 2:
        return "negative"
    elif score == 3:
        return "neutral"
    return "positive"


df["sentiment"] = df["Score"].apply(score_to_sentiment)
df["sentiment"].value_counts(normalize=True)


,proportion
sentiment,
positive,0.779455
negative,0.144971
neutral,0.075574


Функция очистки текста ниже один в один пойдёт в `main.py` (см. Блок 4).
Если она разойдётся с той, что использовалась на обучении — модель в API увидит
другое распределение текста, чем на обучении (train/serve skew), и
просядет без явной ошибки.


In [35]:
import re


def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)          # HTML-теги
    text = re.sub(r"http\S+", " ", text)         # ссылки
    text = re.sub(r"[^a-zA-Z\s]", " ", text)     # только буквы
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


df["clean_text"] = df["Text"].apply(clean_text)
df[["Text", "clean_text"]].head(3)


,Text,clean_text
0,I have bought several of the Vitality canned d...,i have bought several of the vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled as jumbo salted peanut...
2,This is a confection that has been around a fe...,this is a confection that has been around a fe...


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"], test_size=0.2, random_state=42, stratify=df["sentiment"]
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=3)),
    ("clf", LinearSVC(class_weight="balanced", random_state=42)),
])

model.fit(X_train, y_train)
print("Модель обучена.")


Модель обучена.


# Берём 15 реальных отзывов из тестовой выборки

In [10]:
test_negative = df.loc[X_test.index][df.loc[X_test.index, "sentiment"] == "negative"]

delivery = test_negative[test_negative["Text"].str.contains("shipping|delivery|box|package", case=False, na=False)]
taste = test_negative[test_negative["Text"].str.contains("taste|flavor|stale|expired", case=False, na=False)]

demo_reviews = (
    delivery["Text"].sample(5, random_state=1).tolist() +
    taste["Text"].sample(5, random_state=1).tolist() +
    test_negative["Text"].sample(5, random_state=1).tolist()
)

for r in demo_reviews:
    print("-", r[:100])

- As a lover of boxed mac and cheese, I thought that these would be the perfect thing for me, so I bou
- If I could put zero stars I would!<br /><br />I purchased two 64-fl oz containers of pure maple syru
- I tried the Hickory smoked bacon flavor salt to see if it was better than their standard bacon salt.
- I love Ritz crackers.  My complaint is that most of them were broken when I received my shipment.  L
- All the candies in the box had melted somewhat so they had leaked out onto the wrapped paper.  I bou
- We gave this flavor a try but it didn't appeal to our tastes at all. I'm a huge fan or LorAnn's oils
- The chocolate layer on these expresso beans is much thicker, softer, and have a slight cigarette-smo
- Trying to improve my health by eating better. found this whole wheat stuffing and thought, how can y
- If you're gonna put this tween yer cheek n' gum like chewing tobacco you need to know that it will c
- I bought this tea because I have been looking for a good product contai

In [11]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print()
print(classification_report(y_test, y_pred))


Accuracy: 0.8609654857258914
Macro F1: 0.6859838978333683

              precision    recall  f1-score   support

    negative       0.72      0.74      0.73     11421
     neutral       0.36      0.43      0.39      5953
    positive       0.95      0.92      0.94     61405

    accuracy                           0.86     78779
   macro avg       0.68      0.70      0.69     78779
weighted avg       0.87      0.86      0.87     78779



In [12]:
import joblib

joblib.dump(model, "review-api/model.pkl")

size_kb = os.path.getsize("review-api/model.pkl") / 1024
print("Модель сохранена в review-api/model.pkl")
print("Размер файла:", round(size_kb, 1), "КБ")
print("С этой минуты модель — это файл. Она больше не зависит от ноутбука, в котором обучалась.")


Модель сохранена в review-api/model.pkl
Размер файла: 1843.8 КБ
С этой минуты модель — это файл. Она больше не зависит от ноутбука, в котором обучалась.


In [13]:
loaded_model = joblib.load("review-api/model.pkl")

test_review = clean_text("The product arrived broken and tasted awful.")
predicted = loaded_model.predict([test_review])[0]

print("Отзыв: The product arrived broken and tasted awful.")
print("Предсказанная тональность:", predicted)


Отзыв: The product arrived broken and tasted awful.
Предсказанная тональность: negative


In [14]:
import sklearn
import numpy as np

requirements = []
requirements.append("fastapi")
requirements.append("uvicorn[standard]")
requirements.append("openai")
requirements.append("scikit-learn==" + sklearn.__version__)
requirements.append("numpy==" + np.__version__)
requirements.append("joblib==" + joblib.__version__)

with open("review-api/requirements.txt", "w") as f:
    f.write(chr(10).join(requirements))

print(chr(10).join(requirements))
print()
print("scikit-learn, numpy и joblib жёстко зафиксированы — версии, на которых обучалась модель.")
print("fastapi/uvicorn/openai не пиним: на формат model.pkl они не влияют, а устаревший пин ломает сборку.")


fastapi
uvicorn[standard]
openai
scikit-learn==1.6.1
numpy==2.1.3
joblib==1.6.0

scikit-learn, numpy и joblib жёстко зафиксированы — версии, на которых обучалась модель.
fastapi/uvicorn/openai не пиним: на формат model.pkl они не влияют, а устаревший пин ломает сборку.


## Блок 3 — API-ключ для LLM (безопасно, через Colab Secrets)

Не вставляйте ключ прямо в код: если ноутбук случайно попадёт на GitHub,
ключ утечёт вместе с ним. В Colab слева на панели есть иконка ключа 🔑:

1. Нажмите на неё → **Add new secret**
2. Имя: `OPENAI_API_KEY`, значение — ваш ключ
3. Включите переключатель **Notebook access** для этого ноутбука


In [15]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("Ключ подхвачен из Colab Secrets (сам ключ нигде не печатается).")


Ключ подхвачен из Colab Secrets (сам ключ нигде не печатается).


## Блок 4 — FastAPI: у модели и LLM-анализа появляется адрес

In [16]:
%%writefile review-api/llm_analyzer.py
"""
llm_analyzer.py

AI-часть: LLM читает негативные отзывы (уже отфильтрованные ML-моделью),
группирует их по темам жалоб и предлагает бизнесу конкретные улучшения.

Ключ OPENAI_API_KEY ожидается в переменных окружения — в Colab он попадает
туда из Secrets (см. ноутбук), при Docker-запуске — через docker run -e.
"""

import os
import json
from typing import List, Dict

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
DEFAULT_MODEL = "gpt-4o-mini"


PROMPT_TEMPLATE = """Ты — аналитик клиентского опыта для бизнеса, который продаёт продукты питания.
Ниже дана выборка НЕГАТИВНЫХ отзывов покупателей (каждый отзыв пронумерован).

Твоя задача:
1. Сгруппировать отзывы по 3-6 ключевым темам жалоб (например: качество продукта,
   доставка, упаковка, цена, вкус, испорченный товар и т.д.)
2. Для каждой темы:
   - короткое название
   - примерная доля отзывов по теме (%)
   - 1-2 предложения с сутью проблемы
   - один короткий пример (перефразированный, НЕ дословная цитата)
   - одна конкретная, выполнимая рекомендация для бизнеса
3. В конце — общий вывод: топ-3 приоритета для бизнеса.

Ответь СТРОГО в формате JSON, без markdown-разметки и пояснений до/после, по схеме:

{{
  "topics": [
    {{
      "title": "string",
      "share_percent": number,
      "description": "string",
      "example": "string",
      "recommendation": "string"
    }}
  ],
  "top_priorities": ["string", "string", "string"]
}}

Отзывы:
{reviews}
"""


def _format_reviews(reviews: List[str], max_reviews: int = 40) -> str:
    sample = reviews[:max_reviews]
    return "\n".join(f"{i + 1}. {r.strip()}" for i, r in enumerate(sample))


def analyze_negative_reviews(reviews: List[str], model: str = DEFAULT_MODEL) -> Dict:
    if not reviews:
        return {"topics": [], "top_priorities": []}

    prompt = PROMPT_TEMPLATE.format(reviews=_format_reviews(reviews))

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"error": "Не удалось распарсить ответ LLM как JSON", "raw_response": raw}


def format_report_markdown(analysis: Dict) -> str:
    if "error" in analysis:
        return (
            f"⚠️ Ошибка анализа: {analysis['error']}\n\n"
            f"```\n{analysis.get('raw_response', '')}\n```"
        )

    lines = ["# 📊 Отчёт по анализу негативных отзывов\n"]

    for topic in analysis.get("topics", []):
        lines.append(f"## {topic.get('title', 'Без названия')} "
                      f"({topic.get('share_percent', '?')}%)")
        lines.append(f"**Проблема:** {topic.get('description', '')}")
        lines.append(f"**Пример:** _{topic.get('example', '')}_")
        lines.append(f"**Рекомендация:** {topic.get('recommendation', '')}\n")

    if analysis.get("top_priorities"):
        lines.append("## 🎯 Топ-3 приоритета для бизнеса")
        for i, p in enumerate(analysis["top_priorities"], 1):
            lines.append(f"{i}. {p}")

    return "\n".join(lines)


Writing review-api/llm_analyzer.py


In [17]:
%%writefile review-api/main.py
import re
from typing import List

import joblib
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

from llm_analyzer import analyze_negative_reviews, format_report_markdown

app = FastAPI(title="Аналитик отзывов для бизнеса", version="1.0")

# Модель грузится ОДИН раз при старте сервера, а не на каждый запрос
model = joblib.load("model.pkl")


def clean_text(text: str) -> str:
    """Та же функция, что использовалась при обучении модели в ноутбуке —
    если они разойдутся, модель увидит на входе не то, на чём училась."""
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


class Review(BaseModel):
    text: str = Field(..., min_length=1, description="Текст отзыва")


class ReviewBatch(BaseModel):
    reviews: List[str] = Field(..., min_length=1, description="Список текстов отзывов")


@app.get("/")
def index():
    return FileResponse("index.html")


@app.get("/health")
def health():
    return {"status": "healthy"}


@app.get("/info")
def info():
    return {
        "model": "TF-IDF + LinearSVC",
        "classes": ["negative", "neutral", "positive"],
        "version": "1.0",
    }


@app.post("/predict")
def predict(review: Review):
    sentiment = model.predict([clean_text(review.text)])[0]
    return {"text": review.text, "sentiment": sentiment}


@app.post("/predict-batch")
def predict_batch(batch: ReviewBatch):
    cleaned = [clean_text(t) for t in batch.reviews]
    predictions = model.predict(cleaned)
    return {
        "results": [
            {"text": t, "sentiment": s} for t, s in zip(batch.reviews, predictions)
        ]
    }


@app.post("/analyze-negative")
def analyze_negative(batch: ReviewBatch):
    """Сам отбирает негативные отзывы через ML-модель и отправляет их в LLM."""
    cleaned = [clean_text(t) for t in batch.reviews]
    predictions = model.predict(cleaned)
    negative_reviews = [
        t for t, s in zip(batch.reviews, predictions) if s == "negative"
    ]

    if not negative_reviews:
        raise HTTPException(
            status_code=400,
            detail="Среди присланных отзывов не найдено негативных — анализировать нечего.",
        )

    analysis = analyze_negative_reviews(negative_reviews)
    report_md = format_report_markdown(analysis)
    return {
        "negative_count": len(negative_reviews),
        "analysis": analysis,
        "report_markdown": report_md,
    }


Writing review-api/main.py


In [18]:
# @title
%%writefile review-api/index.html
<!doctype html>
<html lang="ru">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Аналитик отзывов</title>
<style>
  :root {
    --ink: #1F2A22;
    --ink-soft: #4B5A4E;
    --bg: #F4F6F0;
    --panel: #FFFFFF;
    --line: #DCE3D6;
    --accent: #3F6C51;
    --accent-soft: #E7EFE4;
    --negative: #B0472E;
    --negative-soft: #F7E7E2;
    --neutral: #A0813F;
    --neutral-soft: #F3ECDC;
    --positive: #3F6C51;
    --positive-soft: #E7EFE4;
    --radius: 10px;
  }

  @media (prefers-color-scheme: dark) {
    :root {
      --ink: #E7ECE4;
      --ink-soft: #AEB8AC;
      --bg: #161B17;
      --panel: #1D231E;
      --line: #2C352D;
      --accent: #7FB893;
      --accent-soft: #22301F;
      --negative: #E58269;
      --negative-soft: #33221D;
      --neutral: #D8B86B;
      --neutral-soft: #302A1B;
      --positive: #7FB893;
      --positive-soft: #22301F;
    }
  }

  * { box-sizing: border-box; }

  body {
    margin: 0;
    background: var(--bg);
    color: var(--ink);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    line-height: 1.5;
  }

  .wrap {
    max-width: 760px;
    margin: 0 auto;
    padding: 48px 24px 80px;
  }

  header h1 {
    font-family: Georgia, "Iowan Old Style", "Palatino Linotype", serif;
    font-size: 2.1rem;
    font-weight: 600;
    margin: 0 0 8px;
    letter-spacing: -0.01em;
  }

  header p {
    margin: 0;
    color: var(--ink-soft);
    max-width: 52ch;
  }

  .panel {
    background: var(--panel);
    border: 1px solid var(--line);
    border-radius: var(--radius);
    padding: 24px;
    margin-top: 28px;
  }

  label {
    display: block;
    font-size: 0.9rem;
    color: var(--ink-soft);
    margin-bottom: 8px;
  }

  textarea {
    width: 100%;
    min-height: 160px;
    padding: 14px;
    border: 1px solid var(--line);
    border-radius: 8px;
    background: var(--bg);
    color: var(--ink);
    font: inherit;
    resize: vertical;
  }

  textarea:focus-visible, button:focus-visible {
    outline: 2px solid var(--accent);
    outline-offset: 2px;
  }

  .actions {
    display: flex;
    gap: 12px;
    margin-top: 16px;
    flex-wrap: wrap;
  }

  button {
    font: inherit;
    font-weight: 600;
    padding: 11px 18px;
    border-radius: 8px;
    border: 1px solid var(--accent);
    background: var(--accent);
    color: #fff;
    cursor: pointer;
  }

  button.secondary {
    background: transparent;
    color: var(--accent);
  }

  button:disabled {
    opacity: 0.6;
    cursor: default;
  }

  button:hover:not(:disabled) {
    filter: brightness(1.05);
  }

  .hint {
    font-size: 0.85rem;
    color: var(--ink-soft);
    margin-top: 10px;
  }

  .results-panel h2, .report-panel h2 {
    font-size: 1.05rem;
    margin: 0 0 16px;
  }

  .review-row {
    display: flex;
    align-items: baseline;
    gap: 10px;
    padding: 10px 0;
    border-bottom: 1px solid var(--line);
  }
  .review-row:last-child { border-bottom: none; }

  .badge {
    font-size: 0.78rem;
    font-weight: 600;
    padding: 3px 9px;
    border-radius: 999px;
    white-space: nowrap;
  }
  .badge.negative { background: var(--negative-soft); color: var(--negative); }
  .badge.neutral  { background: var(--neutral-soft);  color: var(--neutral); }
  .badge.positive { background: var(--positive-soft); color: var(--positive); }

  .review-text {
    color: var(--ink-soft);
    font-size: 0.92rem;
  }

  .topic-card {
    border-top: 1px solid var(--line);
    padding: 18px 0;
  }
  .topic-card:first-of-type { border-top: none; padding-top: 0; }

  .topic-head {
    display: flex;
    justify-content: space-between;
    align-items: baseline;
    gap: 12px;
  }

  .topic-head h3 {
    margin: 0;
    font-size: 1rem;
  }

  .topic-share {
    color: var(--ink-soft);
    font-size: 0.85rem;
    white-space: nowrap;
  }

  .topic-card p {
    margin: 8px 0 0;
    max-width: 68ch;
  }

  .topic-example {
    color: var(--ink-soft);
    font-style: italic;
  }

  .topic-recommendation {
    color: var(--accent);
  }

  .priorities {
    margin: 8px 0 0;
    padding-left: 22px;
  }
  .priorities li { margin-bottom: 6px; }

  .empty-state, .error-state {
    color: var(--ink-soft);
    font-size: 0.92rem;
  }

  .error-state { color: var(--negative); }

  footer {
    margin-top: 40px;
    color: var(--ink-soft);
    font-size: 0.82rem;
  }
</style>
</head>
<body>
<div class="wrap">
  <header>
    <h1>Аналитик отзывов</h1>
    <p>Вставьте отзывы клиентов — модель определит тональность каждого, а на негативных найдёт повторяющиеся темы и предложит, что исправить.</p>
  </header>

  <section class="panel">
    <label for="reviews-input">Отзывы (каждый — с новой строки)</label>
    <textarea id="reviews-input" placeholder="The product arrived damaged and half the jar was leaking.&#10;Great taste, will buy again.&#10;Delivery took three weeks."></textarea>
    <div class="actions">
      <button id="btn-sentiment" onclick="checkSentiment()">Проверить тональность</button>
      <button id="btn-report" class="secondary" onclick="findComplaints()">Найти причины жалоб</button>
    </div>
    <p class="hint">Второй кнопке нужно хотя бы несколько негативных отзывов в списке — по ним LLM собирает отчёт.</p>
  </section>

  <section class="panel results-panel">
    <h2>Тональность</h2>
    <div id="results-body" class="empty-state">Здесь появится тональность каждого отзыва после проверки.</div>
  </section>

  <section class="panel report-panel">
    <h2>Причины жалоб</h2>
    <div id="report-body" class="empty-state">Здесь появится разбивка по темам и рекомендации после анализа.</div>
  </section>

  <footer>Страница обращается к локальному API на этой же машине (FastAPI, порт 8000).</footer>
</div>

<script>
  const sentimentLabels = { negative: 'негатив', neutral: 'нейтрал', positive: 'позитив' };

  function getReviews() {
    const raw = document.getElementById('reviews-input').value;
    return raw.split('\n').map(s => s.trim()).filter(Boolean);
  }

  function setBusy(btnId, busy, label, busyLabel) {
    const btn = document.getElementById(btnId);
    btn.disabled = busy;
    btn.textContent = busy ? busyLabel : label;
  }

  function escapeHtml(str) {
    const div = document.createElement('div');
    div.textContent = str == null ? '' : str;
    return div.innerHTML;
  }

  async function checkSentiment() {
    const reviews = getReviews();
    const body = document.getElementById('results-body');
    if (!reviews.length) {
      body.className = 'empty-state';
      body.textContent = 'Добавьте хотя бы один отзыв в поле выше.';
      return;
    }
    setBusy('btn-sentiment', true, 'Проверить тональность', 'Проверяем…');
    try {
      const res = await fetch('/predict-batch', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ reviews }),
      });
      if (!res.ok) throw new Error('bad-status');
      const data = await res.json();
      renderSentiment(data.results);
    } catch (e) {
      body.className = 'error-state';
      body.textContent = 'Не удалось связаться с API. Убедитесь, что сервер запущен: uvicorn api:app --reload.';
    } finally {
      setBusy('btn-sentiment', false, 'Проверить тональность', 'Проверяем…');
    }
  }

  function renderSentiment(results) {
    const body = document.getElementById('results-body');
    body.className = '';
    body.innerHTML = results.map(r => `
      <div class="review-row">
        <span class="badge ${r.sentiment}">${sentimentLabels[r.sentiment] || r.sentiment}</span>
        <span class="review-text">${escapeHtml(r.text)}</span>
      </div>
    `).join('');
  }

  async function findComplaints() {
    const reviews = getReviews();
    const body = document.getElementById('report-body');
    if (!reviews.length) {
      body.className = 'empty-state';
      body.textContent = 'Добавьте хотя бы один отзыв в поле выше.';
      return;
    }
    setBusy('btn-report', true, 'Найти причины жалоб', 'Анализируем, 10–20 секунд…');
    try {
      const res = await fetch('/analyze-negative', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ reviews }),
      });
      if (res.status === 400) {
        const err = await res.json();
        body.className = 'empty-state';
        body.textContent = err.detail || 'Среди присланных отзывов не найдено негативных.';
        return;
      }
      if (!res.ok) throw new Error('bad-status');
      const data = await res.json();
      renderReport(data.analysis);
    } catch (e) {
      body.className = 'error-state';
      body.textContent = 'Не удалось связаться с API. Убедитесь, что сервер запущен: uvicorn api:app --reload.';
    } finally {
      setBusy('btn-report', false, 'Найти причины жалоб', 'Анализируем, 10–20 секунд…');
    }
  }

  function renderReport(analysis) {
    const body = document.getElementById('report-body');
    if (!analysis || analysis.error) {
      body.className = 'error-state';
      body.textContent = 'LLM вернула ответ, который не удалось разобрать. Попробуйте ещё раз.';
      return;
    }
    body.className = '';
    const topics = (analysis.topics || []).map(t => `
      <div class="topic-card">
        <div class="topic-head">
          <h3>${escapeHtml(t.title)}</h3>
          <span class="topic-share">${t.share_percent ?? '?'}%</span>
        </div>
        <p>${escapeHtml(t.description)}</p>
        <p class="topic-example">«${escapeHtml(t.example)}»</p>
        <p class="topic-recommendation">${escapeHtml(t.recommendation)}</p>
      </div>
    `).join('');

    const priorities = (analysis.top_priorities || []).map(p => `<li>${escapeHtml(p)}</li>`).join('');

    body.innerHTML = topics + (priorities ? `
      <div class="topic-card">
        <h3>Топ-приоритеты</h3>
        <ol class="priorities">${priorities}</ol>
      </div>
    ` : '');
  }
</script>
</body>
</html>


Writing review-api/index.html


## Блок 5 — Запускаем сервер внутри Colab

In [19]:
server_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="review-api",
)
time.sleep(10)

print("Сервер запущен в фоне, PID:", server_process.pid)
print("Адрес внутри этой машины: http://127.0.0.1:8000")


Сервер запущен в фоне, PID: 5282
Адрес внутри этой машины: http://127.0.0.1:8000


In [20]:
import requests

BASE_URL = "http://127.0.0.1:8000"

health_response = requests.get(BASE_URL + "/health", timeout=10)
print("Код ответа:", health_response.status_code)
print("Тело ответа:", health_response.json())


Код ответа: 200
Тело ответа: {'status': 'healthy'}


In [21]:
sample_reviews = [
    "The product arrived damaged and half the jar was leaking everywhere.",
    "Great taste, will buy again, best purchase this year.",
    "It's fine, nothing special but does the job.",
    "Delivery took three weeks and the box was falling apart.",
    "Way too expensive for what you actually get in the jar.",
]

response = requests.post(BASE_URL + "/predict-batch", json={"reviews": sample_reviews}, timeout=10)
print("Код ответа:", response.status_code)
for r in response.json()["results"]:
    print(r["sentiment"].ljust(10), "-", r["text"][:70])


Код ответа: 200
negative   - The product arrived damaged and half the jar was leaking everywhere.
positive   - Great taste, will buy again, best purchase this year.
neutral    - It's fine, nothing special but does the job.
negative   - Delivery took three weeks and the box was falling apart.
negative   - Way too expensive for what you actually get in the jar.


Теперь — AI-часть. Модель сама отберёт негативные отзывы из списка,
а LLM разберёт их на темы и предложит рекомендации. Это может занять
10-20 секунд — идёт настоящий запрос к OpenAI.


In [28]:
response = requests.post(BASE_URL + "/analyze-negative", json={"reviews": sample_reviews}, timeout=60)
print("Код ответа:", response.status_code)

result = response.json()
print("Негативных отзывов найдено:", result["negative_count"])
print()
print(result["report_markdown"])


Код ответа: 200
Негативных отзывов найдено: 3

# 📊 Отчёт по анализу негативных отзывов

## Качество продукта (33%)
**Проблема:** Покупатели жалуются на поврежденные товары, которые приходят в ненадлежащем состоянии.
**Пример:** _Товар пришел с поврежденной упаковкой и утечкой._
**Рекомендация:** Убедитесь в качестве упаковки и проведите дополнительные проверки перед отправкой.

## Доставка (33%)
**Проблема:** Долгие сроки доставки и ненадежная упаковка вызывают недовольство у клиентов.
**Пример:** _Доставка заняла слишком много времени, и коробка была в плохом состоянии._
**Рекомендация:** Оптимизируйте логистику и используйте более прочные упаковочные материалы.

## Цена (34%)
**Проблема:** Клиенты считают цены завышенными по сравнению с качеством продукта.
**Пример:** _Цена слишком высокая для объема и качества товара._
**Рекомендация:** Пересмотрите ценовую политику и предложите более конкурентоспособные цены.

## 🎯 Топ-3 приоритета для бизнеса
1. Качество продукта
2. Доставка
3. Це

Проверим и обратный случай — что будет, если прислать данные не по схеме.
Ни одной строчки проверок мы не писали: их сгенерировал Pydantic из описания
класса `ReviewBatch` в `main.py`. Код `422` означает «данные не прошли
валидацию», и сервер возвращает понятное объяснение вместо падения.


In [23]:
bad_requests = [
    {"reviews": []},
    {"text": "тут не то поле"},
]

for bad in bad_requests:
    r = requests.post(BASE_URL + "/predict-batch", json=bad, timeout=10)
    print("Отправили:", bad)
    print("Код ответа:", r.status_code)
    print()


Отправили: {'reviews': []}
Код ответа: 422

Отправили: {'text': 'тут не то поле'}
Код ответа: 422



## Блок 6 — Смотрим интерфейс прямо в Colab

In [24]:
try:
    from google.colab.output import serve_kernel_port_as_iframe
    serve_kernel_port_as_iframe(8000, height=650)
except ImportError:
    print("Вы не в Colab. Откройте в браузере http://127.0.0.1:8000")


<IPython.core.display.Javascript object>

## Блок 7 — Docker


In [26]:
%%writefile review-api/Dockerfile
# Базовый образ: Python 3.11 в облегчённой сборке
FROM python:3.11-slim

WORKDIR /app

# Сначала только зависимости — Docker закеширует этот слой
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Теперь код, интерфейс и сама модель
COPY main.py .
COPY llm_analyzer.py .
COPY index.html .
COPY model.pkl .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]


Writing review-api/Dockerfile


In [27]:
%%writefile review-api/.dockerignore
__pycache__/
*.pyc
.git/
.env
*.log
*.ipynb
data/


Writing review-api/.dockerignore


In [38]:
EC2_IP = '54.252.232.152'
CLOUD_URL = 'http://' + EC2_IP + ':8000'

print('Обращаемся к серверу в облаке:', CLOUD_URL)
print('Запрос уйдёт из Астаны в дата-центр Amazon и вернётся обратно.')

Обращаемся к серверу в облаке: http://54.252.232.152:8000
Запрос уйдёт из Астаны в дата-центр Amazon и вернётся обратно.


In [39]:
try:
    cloud_health = requests.get(CLOUD_URL + '/health', timeout=10)
    print('Код ответа:', cloud_health.status_code)
    print('Тело ответа:', cloud_health.json())
except Exception as error:
    print('Не достучались:', error)
    print('Проверьте: верный ли IP, открыт ли порт 8000 в Security Group, запущен ли контейнер.')

Код ответа: 200
Тело ответа: {'status': 'healthy'}


In [40]:
sample_reviews = [
    "The product arrived damaged and half the jar was leaking everywhere.",
    "Great taste, will buy again, best purchase this year.",
    "Delivery took three weeks and the box was falling apart.",
]

cloud_answer = requests.post(CLOUD_URL + '/predict-batch', json={"reviews": sample_reviews}, timeout=10).json()

for r in cloud_answer['results']:
    print(r['sentiment'].ljust(10), '-', r['text'][:70])

print()
print('Тот же адрес (http://54.252.232.152:8000) можно открыть в браузере телефона — откроется index.html с формой.')

negative   - The product arrived damaged and half the jar was leaking everywhere.
positive   - Great taste, will buy again, best purchase this year.
negative   - Delivery took three weeks and the box was falling apart.

Тот же адрес (http://ВАШ_IP:8000) можно открыть в браузере телефона — откроется index.html с формой.


In [41]:
latencies = []

for i in range(20):
    started = time.time()
    requests.post(CLOUD_URL + '/predict-batch', json={"reviews": sample_reviews}, timeout=10)
    latencies.append((time.time() - started) * 1000)

print('Среднее время ответа:', round(np.mean(latencies)), 'мс')
print('Минимум:', round(min(latencies)), 'мс')
print('Максимум:', round(max(latencies)), 'мс')

Среднее время ответа: 237 мс
Минимум: 226 мс
Максимум: 271 мс


In [42]:
started = time.time()
report = requests.post(CLOUD_URL + '/analyze-negative', json={"reviews": sample_reviews}, timeout=60).json()
elapsed = (time.time() - started) * 1000

print('Время ответа AI-эндпоинта:', round(elapsed), 'мс')
print()
print(report['report_markdown'])

Время ответа AI-эндпоинта: 3143 мс

# 📊 Отчёт по анализу негативных отзывов

## Качество упаковки (50%)
**Проблема:** Покупатели жалуются на плохое состояние упаковки, что приводит к повреждениям товаров во время доставки.
**Пример:** _Упаковка товара была в ужасном состоянии, и часть продукта вылилась._
**Рекомендация:** Улучшить качество упаковки, используя более прочные материалы и дополнительные защитные элементы.

## Сроки доставки (50%)
**Проблема:** Долгие сроки доставки вызывают недовольство у клиентов, что негативно сказывается на их опыте.
**Пример:** _Доставка заняла три недели, что крайне неудовлетворительно._
**Рекомендация:** Оптимизировать логистику и установить более реалистичные сроки доставки для клиентов.

## 🎯 Топ-3 приоритета для бизнеса
1. Улучшение качества упаковки
2. Оптимизация сроков доставки
3. Повышение контроля качества товаров


## Останавливаем локальный сервер в Colab

In [34]:
server_process.terminate()
time.sleep(2)
print("Локальный сервер в Colab остановлен.")


Локальный сервер в Colab остановлен.
